In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

API_KEY = input("Paste your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

In [ ]:
MODEL_NAME = "gemini-3.5-flash-lite"

SYSTEM_TEXT = """
You are a healthcare appointment follow-up analyst.
Classify patient call-centre notes into one root cause of missed appointments.

Allowed root causes:
forgot_or_no_reminder,
distance_or_transport,
cost_or_fee,
illness_or_emergency,
work_or_schedule,
other.

Use only the allowed root cause values.
Do not guess when the note is ambiguous. Choose other if the reason is unclear.
Return accurate, concise and structured information.
"""

In [ ]:
def analyse_note(note_text):

    prompt = f"""
    Context: CareLoop clinics wants to understand why patients miss appointments.
    Role: You are a healthcare appointment follow-up analyst.
    Action: Classify the note into exactly one allowed root cause.
    Allowed values: forgot_or_no_reminder, distance_or_transport,
    cost_or_fee, illness_or_emergency, work_or_schedule, other.
    If the reason is unclear, choose other instead of guessing.
    Tone: Neutral, factual and concise.
    Format: Return root_cause, confidence (0 to 1),
    summary (maximum 25 words), and recommended_action.

    Patient note: {note_text}
    """
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
            thinking_level="minimal"

            )
        )
    )

    data = json.loads(response.text)
    return data

In [ ]:
note_n001 = "Patient forgot the appointment and did not receive a reminder."

result_n001 = analyse_note(note_n001)

print(result_n001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient forgot the appointment because they did not receive a reminder.', 'recommended_action': 'Enable automated SMS and email reminders for future appointments.'}


In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

In [ ]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [ ]:
import time

notes_df = pd.read_csv("followup_notes.csv")

results = []

for _, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"Processing {note_id}...")

    # Blank note handling
    if pd.isna(note_text) or str(note_text).strip() == "":
        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })
        print("Skipped blank note")
        continue

    try:
        raw_data = analyse_note(str(note_text))
        raw_response = json.dumps(raw_data, ensure_ascii=False)

        validated = FollowupInsight(**raw_data)

        if validated.root_cause not in ALLOWED_CAUSES:
            raise ValueError("Invalid root cause category")

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": validated.root_cause,
            "confidence": validated.confidence,
            "summary": validated.summary,
            "recommended_action": validated.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_response
        })

        print("Validated successfully")

    except Exception as e:

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": raw_data.get("root_cause", "") if "raw_data" in locals() else "",
            "confidence": raw_data.get("confidence", "") if "raw_data" in locals() else "",
            "summary": raw_data.get("summary", "") if "raw_data" in locals() else "",
            "recommended_action": raw_data.get("recommended_action", "") if "raw_data" in locals() else "",
            "validation_status": "invalid",
            "raw_response": raw_response if "raw_response" in locals() else ""
        })

        print("Validation/API error:", e)

    time.sleep(13)

results_df = pd.DataFrame(results)

results_df.to_csv("notes_analysed.csv", index=False)

display(results_df)

Processing N001...
Validated successfully
Processing N002...
Validated successfully
Processing N003...
Validated successfully
Processing N004...
Skipped blank note
Processing N005...
Validated successfully
Processing N006...
Validated successfully
Processing N007...
Validated successfully
Processing N008...
Validated successfully
Processing N009...
Validated successfully
Processing N010...
Validated successfully


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,1.00,Patient missed the appointment because she did...,Verify and update patient contact information ...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi..."
1,N002,A0897,distance_or_transport,1.00,Patient missed the appointment because they li...,Offer telehealth options or assistance with lo...,valid,"{""root_cause"": ""distance_or_transport"", ""confi..."
2,N003,A0526,cost_or_fee,1.00,Patient skipped the appointment due to high co...,"Provide information on sliding-scale fees, fin...",valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 1...."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient co...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid..."
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening slot...,valid,"{""root_cause"": ""work_or_schedule"", ""confidence..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because they di...,Reschedule the appointment and verify the pati...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi..."
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Provide information on accessible transportati...,valid,"{""root_cause"": ""distance_or_transport"", ""confi..."
8,N009,A0977,other,0.50,Patient could not be reached after three attem...,Send an automated text message or mailer to re...,valid,"{""root_cause"": ""other"", ""confidence"": 0.5, ""su..."
9,N010,A0596,work_or_schedule,0.95,Patient missed the appointment due to an unexp...,Reschedule the appointment for a Saturday and ...,valid,"{""root_cause"": ""work_or_schedule"", ""confidence..."


In [ ]:
print("Total notes processed:", len(results_df))
print("\nValidation status:")
print(results_df["validation_status"].value_counts())

print("\nRoot cause counts:")
print(results_df["root_cause"].value_counts())

print("\nInvalid root causes:")
print(
    results_df.loc[
        ~results_df["root_cause"].isin(ALLOWED_CAUSES),
        ["note_id", "root_cause"]
    ]
)

print("\nCSV saved:", "notes_analysed.csv")

Total notes processed: 10

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root cause counts:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Invalid root causes:
Empty DataFrame
Columns: [note_id, root_cause]
Index: []

CSV saved: notes_analysed.csv


In [ ]:
from pydantic import ValidationError

# Test 1: Invalid confidence
try:
    FollowupInsight(
        root_cause="other",
        confidence=1.7,
        summary="Reason unclear",
        recommended_action="Call patient again"
    )
except ValidationError as e:
    print("Test 1 Passed: Invalid confidence rejected")
    print(e)

# Test 2: Missing field
try:
    FollowupInsight(
        root_cause="other",
        confidence=0.5,
        summary="Reason unclear"
    )
except ValidationError as e:
    print("\nTest 2 Passed: Missing field rejected")
    print(e)

Test 1 Passed: Invalid confidence rejected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2 Passed: Missing field rejected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...mary': 'Reason unclear'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
def simple_prompt_analysis(note_text):

    prompt = f"Classify this note and return the required JSON:\n{note_text}"

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)


comparison_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]

comparison_results = []

for _, row in comparison_notes.iterrows():

    note_id = row["note_id"]
    note_text = str(row["note_text"])

    print("Comparing", note_id)

    # V1: Simple prompt
    v1 = simple_prompt_analysis(note_text)

    time.sleep(13)

    # V2: CRAFT prompt
    v2 = analyse_note(note_text)

    comparison_results.append({
        "note_id": note_id,
        "V1_root_cause": v1["root_cause"],
        "V1_valid": v1["root_cause"] in ALLOWED_CAUSES,
        "V2_root_cause": v2["root_cause"],
        "V2_valid": v2["root_cause"] in ALLOWED_CAUSES
    })

    time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

Comparing N007
Comparing N009


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
print(comparison_results)

[{'note_id': 'N007', 'V1_root_cause': 'forgot_or_no_reminder', 'V1_valid': True, 'V2_root_cause': 'forgot_or_no_reminder', 'V2_valid': True}]


In [ ]:
note_n009 = str(
    notes_df.loc[notes_df["note_id"] == "N009", "note_text"].iloc[0]
)

v2_n009 = analyse_note(note_n009)

comparison_results.append({
    "note_id": "N009",
    "V1_root_cause": v1["root_cause"],
    "V1_valid": v1["root_cause"] in ALLOWED_CAUSES,
    "V2_root_cause": v2_n009["root_cause"],
    "V2_valid": v2_n009["root_cause"] in ALLOWED_CAUSES
})

display(pd.DataFrame(comparison_results))

,note_id,V1_root_cause,V1_valid,V2_root_cause,V2_valid
0,N007,forgot_or_no_reminder,True,forgot_or_no_reminder,True
1,N009,other,True,other,True


In [ ]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df.to_csv("prompt_comparison.csv", index=False)

display(comparison_df)

,note_id,V1_root_cause,V1_valid,V2_root_cause,V2_valid
0,N007,forgot_or_no_reminder,True,forgot_or_no_reminder,True
1,N009,other,True,other,True


In [ ]:
appointments_df = pd.read_csv("appointments_clean.csv")
notes_analysis_df = pd.read_csv("notes_analysed.csv")

merged_df = notes_analysis_df.merge(
    appointments_df[["appointment_id", "reminder_sent", "no_show"]],
    on="appointment_id",
    how="left"
)

display(merged_df.head())

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,reminder_sent,no_show
0,N001,A0872,forgot_or_no_reminder,1.00,Patient missed the appointment because she did...,Verify and update patient contact information ...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi...",No,1
1,N002,A0897,distance_or_transport,1.00,Patient missed the appointment because they li...,Offer telehealth options or assistance with lo...,valid,"{""root_cause"": ""distance_or_transport"", ""confi...",No,1
2,N003,A0526,cost_or_fee,1.00,Patient skipped the appointment due to high co...,"Provide information on sliding-scale fees, fin...",valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 1....",Yes,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,No,1
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient co...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid...",Yes,1


In [ ]:
appointments_df = pd.read_csv("appointments_clean.csv")

merged_df = results_df.merge(
    appointments_df[["appointment_id", "reminder_sent", "no_show"]],
    on="appointment_id",
    how="left"
)

display(merged_df.head())

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,reminder_sent,no_show
0,N001,A0872,forgot_or_no_reminder,1.00,Patient missed the appointment because she did...,Verify and update patient contact information ...,valid,"{""root_cause"": ""forgot_or_no_reminder"", ""confi...",No,1
1,N002,A0897,distance_or_transport,1.00,Patient missed the appointment because they li...,Offer telehealth options or assistance with lo...,valid,"{""root_cause"": ""distance_or_transport"", ""confi...",No,1
2,N003,A0526,cost_or_fee,1.00,Patient skipped the appointment due to high co...,"Provide information on sliding-scale fees, fin...",valid,"{""root_cause"": ""cost_or_fee"", ""confidence"": 1....",Yes,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,,No,1
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Reschedule the appointment once the patient co...,valid,"{""root_cause"": ""illness_or_emergency"", ""confid...",Yes,1


In [ ]:
reminder_analysis = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    margins=True
)

display(reminder_analysis)

reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10


In [ ]:
cause_no_show = pd.crosstab(
    merged_df["root_cause"],
    merged_df["no_show"],
    margins=True
)

display(cause_no_show)

no_show,1,All
root_cause,,
cost_or_fee,1,1
distance_or_transport,2,2
forgot_or_no_reminder,2,2
illness_or_emergency,1,1
other,2,2
work_or_schedule,2,2
All,10,10


In [ ]:
merged_df.to_csv("careloop_genai_merged.csv", index=False)

print("Merged dataset saved successfully.")
print("Total records:", len(merged_df))
print("Unique appointments:", merged_df["appointment_id"].nunique())

Merged dataset saved successfully.
Total records: 10
Unique appointments: 10


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

winner_model.fit(X_train, y_train)

test_prob = winner_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)

print("Selected Threshold:", chosen_threshold)
print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))
print("Precision:", round(precision_score(y_test, test_pred), 4))
print("Recall:", round(recall_score(y_test, test_pred), 4))
print("F1:", round(f1_score(y_test, test_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, test_prob), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

Selected Threshold: 0.35
Accuracy: 0.7879
Precision: 0.5714
Recall: 0.5714
F1: 0.5714
ROC-AUC: 0.7889
Confusion Matrix:
[[128  21]
 [ 21  28]]


In [ ]:
risk_scores_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "actual_no_show": y_test.values,
    "no_show_probability": test_prob,
    "predicted_no_show": test_pred
})

risk_scores_df = risk_scores_df.sort_values(
    "no_show_probability", ascending=False
)

risk_scores_df.to_csv("CareLoop_test_risk_scores.csv", index=False)

display(risk_scores_df.head(10))

print("Risk scores saved successfully.")

,appointment_id,actual_no_show,no_show_probability,predicted_no_show
68,A0635,1,0.937204,1
24,A0544,1,0.924066,1
141,A0293,1,0.919634,1
106,A0950,1,0.818616,1
114,A0996,0,0.766710,1
69,A0067,0,0.749805,1
172,A0769,1,0.744427,1
10,A0364,0,0.731920,1
160,A0806,1,0.703462,1
173,A0560,0,0.701898,1


Risk scores saved successfully.
